In [1]:
import os
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langfuse.langchain import CallbackHandler # for LangChain callbacks, like tracking events
from pathlib import Path
from pydantic import SecretStr
from rag_tutorial.rag_utils import print_wrapped

C:\Users\goran\AppData\Local\Temp\ipykernel_22604\2435696253.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [2]:
load_dotenv()
langfuse_handler = CallbackHandler()

Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 


In [3]:

api_key = SecretStr(os.getenv("OPENROUTER_API_KEY", ""))
if not api_key.get_secret_value():
    raise RuntimeError("OPENROUTER_API_KEY is not set")


## Schritt 1: Dokumente laden und chunken

In [4]:
# Create sample files for testing

# Projektstamm anhand von pyproject.toml finden, unabhängig vom Arbeitsverzeichnis des Notebooks
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print(f"Project root: {project_root}")
documents_dir = project_root / "documents"
documents_dir.mkdir(exist_ok=True)
print(f"Documents dir: {documents_dir}")

Project root: c:\Artificial Intelligence\rag_tutorial
Documents dir: c:\Artificial Intelligence\rag_tutorial\documents


In [5]:
loader = DirectoryLoader(
    str(documents_dir),
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

In [6]:
raw_documents = loader.load()

In [7]:
type(raw_documents[0])

langchain_core.documents.base.Document

In [8]:
for doc in raw_documents:
    source = doc.metadata.get("source", "<unknown>")
    print(f"{source + ':':<28} {len(doc.page_content)} characters")

c:\Artificial Intelligence\rag_tutorial\documents\ai-intro.txt: 30133 characters
c:\Artificial Intelligence\rag_tutorial\documents\encoding_problems.txt: 593 characters
c:\Artificial Intelligence\rag_tutorial\documents\messy_report.txt: 609 characters
c:\Artificial Intelligence\rag_tutorial\documents\ml-practical-guide.txt: 12221 characters
c:\Artificial Intelligence\rag_tutorial\documents\neural-networks.txt: 11125 characters
c:\Artificial Intelligence\rag_tutorial\documents\python-classes.txt: 3218 characters
c:\Artificial Intelligence\rag_tutorial\documents\python-functions.txt: 2619 characters
c:\Artificial Intelligence\rag_tutorial\documents\python-intro.txt: 3849 characters
c:\Artificial Intelligence\rag_tutorial\documents\scraped_content.txt: 693 characters
c:\Artificial Intelligence\rag_tutorial\documents\short.txt: 59 characters


In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

In [10]:
documents = text_splitter.split_documents(raw_documents)

In [11]:
print(f"First document:\n{documents[0].page_content}\n \ndocument metadata:\n{documents[0].metadata}")

First document:
# Introduction to AI and ML

## I. Definitions and Conceptual Foundations

Artificial Intelligence (AI) refers to the interdisciplinary scientific field
devoted to creating machines capable of performing tasks that typically require
human intelligence. These tasks encompass perception (computer vision, speech
recognition), reasoning (logical inference, problem-solving), learning (pattern
recognition, adaptation), and decision-making under uncertainty. However, the
definition of AI remains fluid; as computers master specific capabilities—such
as arithmetic, chess, or facial recognition—these tasks often cease to be
classified as "artificial intelligence" in popular discourse and instead become
mere "computation." This phenomenon, known as the **AI effect**, suggests that
intelligence is frequently defined as whatever machines cannot yet do.
 
document metadata:
{'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\ai-intro.txt'}



## Schritt 2: Vektor-Store erstellen

- Wir nutzen Qdrant für die Vektor-Speicherung

In [12]:
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

In [13]:
embeddings = OpenAIEmbeddings(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    model="openai/text-embedding-3-small"
)

In [14]:
vectorstore = QdrantVectorStore.from_documents(
    documents=documents,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    collection_name="ml_docs",
    location=":memory:",
    retrieval_mode=RetrievalMode.HYBRID,
)


## Schritt 3: Retriever erstellen

- Der **Retriever** durchsucht den Vektor-Store
- Gibt die `k` ähnlichsten Dokumente zurück
- Wird aus dem Vektor-Store erstellt

In [15]:
""" Set up retriever for search """
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [16]:
# finde relevante docs
docs_found = retriever.invoke("What is overfitting?")

In [17]:
for i, doc in enumerate(docs_found, 1):
    doc_source = doc.metadata.get("source", "<unknown>")
    print(f"\n{i}. {doc_source}\n   {doc.page_content[:150]}...")


1. c:\Artificial Intelligence\rag_tutorial\documents\ml-practical-guide.txt
   ## Overfitting and Underfitting

Overfitting occurs when a model learns the training data too well, capturing
not just the underlying patterns but als...

2. c:\Artificial Intelligence\rag_tutorial\documents\ml-practical-guide.txt
   The tension between overfitting and underfitting reflects the fundamental
bias-variance tradeoff in machine learning. Simpler models exhibit high bias...



## Schritt 4: Dokumente formatieren

- Der Retriever gibt **Document-Objekte** zurück
- Das LLM braucht aber **normalen Text**
- `format_docs` verbindet alle Dokumente zu einem Text

In [18]:
def format_docs(docs, prefix="=====\n")->str:
    return prefix + f"\n\n{prefix}".join(doc.page_content for doc in docs)

example_docs = retriever.invoke("What is overfitting?")
formatted = format_docs(example_docs)
print(formatted)


=====
## Overfitting and Underfitting

Overfitting occurs when a model learns the training data too well, capturing
not just the underlying patterns but also the noise and random fluctuations
specific to the training set. An overfitted model performs excellently on
training data but poorly on new, unseen data -- it has memorized rather than
generalized. The classic sign of overfitting is a large gap between training
performance (high) and validation or test performance (significantly lower).

Underfitting is the opposite problem: the model is too simple to capture the
underlying patterns in the data, performing poorly on both training and test
data. An underfitting model has not learned enough from the training data to
make useful predictions. This typically occurs when the model lacks sufficient
capacity (too few parameters or too simple an architecture) or when training
has not proceeded long enough.

=====
The tension between overfitting and underfitting reflects the fundamental
bia


## Schritt 5: Retrieval-Chain bauen

Mit LCEL (LangChain Expression Language) können wir Schritte verketten:

In [19]:
# Retrieval-Chain aufbauen, retriever + format_docs, dann ausgeben. retriever holt docs
# format_docs wandelt sie in Text um
retrieval_chain = retriever | format_docs
context_text = retrieval_chain.invoke("What is overfitting?")
print(context_text)

=====
## Overfitting and Underfitting

Overfitting occurs when a model learns the training data too well, capturing
not just the underlying patterns but also the noise and random fluctuations
specific to the training set. An overfitted model performs excellently on
training data but poorly on new, unseen data -- it has memorized rather than
generalized. The classic sign of overfitting is a large gap between training
performance (high) and validation or test performance (significantly lower).

Underfitting is the opposite problem: the model is too simple to capture the
underlying patterns in the data, performing poorly on both training and test
data. An underfitting model has not learned enough from the training data to
make useful predictions. This typically occurs when the model lacks sufficient
capacity (too few parameters or too simple an architecture) or when training
has not proceeded long enough.

=====
The tension between overfitting and underfitting reflects the fundamental
bia


## Schritt 6: Prompt mit Kontext

Wir erstellen einen Prompt, der den Kontext und die Frage enthält:


## Schritt 7: Chat-Modell erstellen

In [20]:
qwen = "qwen/qwen3-30b-a3b-instruct-2507"
llm = ChatOpenRouter(
    api_key=api_key,
    model=qwen,
    temperature=0,
)

## Schritt 8: Die vollständige RAG Chain

<img src="./img/rag-chain-de.png" alt="RAG Chain"
     style="display: block; margin: auto; width: 35%;"/>


## Zusammenfassung

1. embeddings und sparse werden genutzt
2. datenbank QdrantVectorStore wird genutzt
3. retriever sucht passende Dokumente
4. Formatierung da documents sind von type Document, wir brauchen str für LLM
5. retriever chain bauen für LLM
 Prompt mit Kontext erstellen
7. Chat-Modell erstellen
8. Vollständige RAG Chain zusammenbauen

In [21]:
embeddings = OpenAIEmbeddings(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    model="openai/text-embedding-3-small"
)

sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

In [22]:
vectorstore = QdrantVectorStore.from_documents(
    documents=documents,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    collection_name="ml_docs",
    location=":memory:",
    retrieval_mode=RetrievalMode.HYBRID,
)

## Retrieval-Parameter: `k`

- **k**: Wie viele Dokumente abrufen?
  - Mehr → mehr Kontext, aber auch mehr Rauschen
  - Weniger → fokussierter, aber evtl. Information fehlt

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2}) # get top 2 docs, for each question

In [24]:
def format_docs(docs, prefix="=====\n")->str:
    return prefix + f"\n\n{prefix}".join(doc.page_content for doc in docs)

In [25]:
retrieval_chain = retriever | format_docs

In [26]:
system_prompt = (
    "Du bist ein hilfreicher Assistent. Nutze den folgenden Kontext, "
    "um die Frage zu beantworten. "
    "Wenn der Kontext keine relevanten Informationen enthält, sage: "
    "'Diese Information ist nicht in meinen Dokumenten enthalten.' "
    "Halte die Antwort kurz und prägnant.\n\n"
    "Kontext:\n\n{context}"
)

In [27]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [28]:
MINISTRAL = "qwen/qwen3-30b-a3b-instruct-2507"
llm = ChatOpenRouter(
    api_key=api_key,
    model=MINISTRAL,
    temperature=0,
)


## Die Chain Schritt für Schritt

1. Die Frage geht an **zwei** Stellen gleichzeitig:
   - `retrieval_chain` → findet relevante Dokumente → Text
   - `RunnablePassthrough()` → gibt die Frage unverändert weiter
2. Beides wird in den **Prompt** eingesetzt (`{context}` und `{input}`)
3. Der Prompt geht ans **LLM**
4. `StrOutputParser()` extrahiert den Text aus der LLM-Antwort

In [29]:
rag_chain = (
    {"context": retrieval_chain, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
).with_config(callbacks=[langfuse_handler])

### Anfrage (bei jeder Frage)
9. RAG-System nutzen — **R**etrieval, **A**ugmentation, **G**eneration

In [30]:
question = "What is overfitting?"
answer = rag_chain.invoke(question)
print(answer)

Overfitting occurs when a model learns the training data too well, capturing not only the underlying patterns but also the noise and random fluctuations specific to the training set. As a result, it performs excellently on the training data but poorly on new, unseen data because it has memorized rather than generalized.


In [31]:
print_wrapped(answer)

Overfitting occurs when a model learns the training data too well, capturing not
only the underlying patterns but also the noise and random fluctuations specific
to the training set. As a result, it performs excellently on the training data
but poorly on new, unseen data because it has memorized rather than generalized.


In [32]:
# Die Quellen bekommen wir separat vom Retriever:
sources = retriever.invoke(question)
for i, doc in enumerate(sources, 1):
    print(f"{i}. {doc.page_content}...")

1. ## Overfitting and Underfitting

Overfitting occurs when a model learns the training data too well, capturing
not just the underlying patterns but also the noise and random fluctuations
specific to the training set. An overfitted model performs excellently on
training data but poorly on new, unseen data -- it has memorized rather than
generalized. The classic sign of overfitting is a large gap between training
performance (high) and validation or test performance (significantly lower).

Underfitting is the opposite problem: the model is too simple to capture the
underlying patterns in the data, performing poorly on both training and test
data. An underfitting model has not learned enough from the training data to
make useful predictions. This typically occurs when the model lacks sufficient
capacity (too few parameters or too simple an architecture) or when training
has not proceeded long enough....
2. The tension between overfitting and underfitting reflects the fundamental
bias-va

In [33]:
questions = [
    "How do neural networks work?",
    "What is RAG?",
    "Explain gradient descent",
    "What is the best recipe for chocolate cake?" # irrelevant
]

In [34]:
for q in questions:
    answer = rag_chain.invoke(q)
    print(f"\nQ: {q}")
    print(f"A: {answer}")


Q: How do neural networks work?
A: Neural networks are computational models inspired by biological brains that function as mathematical function approximators. They consist of layers of interconnected neurons. Each neuron receives inputs, multiplies them by weights, adds a bias, and applies an activation function to produce an output. Data flows from the input layer through one or more hidden layers to the output layer, where predictions are made. The network learns by adjusting weights and biases during training to minimize prediction errors.

Q: What is RAG?
A: Diese Information ist nicht in meinen Dokumenten enthalten.

Q: Explain gradient descent
A: Gradient descent is an optimization algorithm used to minimize the loss function in neural networks. It works by iteratively updating each weight in the direction opposite to the gradient of the loss with respect to that weight. The size of the update is controlled by the learning rate. A well-chosen learning rate ensures efficient con


## Retrieval-Parameter: `search_type`

**search_type**:
- `"mmr"` (Maximal Marginal Relevance): Bevorzugt **diverse** Ergebnisse —
  nützlich, wenn die ähnlichsten Dokumente sich zu sehr überlappen

In [35]:
retriever_k5 = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20}) # MMR mit k=5, diverse Ergebnisse, fetch_k=20 bedeutet mehr Kandidaten holen

docs_k5 = retriever_k5.invoke(question)

for i, doc in enumerate(docs_k5, 1):
    print(f"  {i}. {doc.page_content[:200]}...")

  1. ## Overfitting and Underfitting

Overfitting occurs when a model learns the training data too well, capturing
not just the underlying patterns but also the noise and random fluctuations
specific to th...
  2. Regularization techniques constrain model complexity to reduce overfitting
without sacrificing the model's ability to learn meaningful patterns. L1
regularization (also known as Lasso) adds a penalty ...
  3. Transfer learning has become a standard practice that dramatically reduces the
data and compute required for specific tasks. Rather than training a neural
network from scratch, practitioners start wit...
  4. Cross-validation provides a robust estimate of model performance by repeatedly
splitting the data into training and validation portions. In k-fold
cross-validation, the data is divided into k equal pa...
  5. The train/test split is a fundamental practice for honest evaluation. Before any
model training begins, a portion of the data (typically 20-30%) is set aside a

- `"similarity_score_threshold"`: Alle Dokumente über einem
  Ähnlichkeits-Schwellenwert

In [36]:
retriever_threshold = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.6, "k": 10} # set threshold, e.g. 0.6
)
docs_threshold = retriever_threshold.invoke(question)

for i, doc in enumerate(docs_threshold, 1):
    print(f"  {i}. {doc.page_content[:200]}...")

  1. ## Overfitting and Underfitting

Overfitting occurs when a model learns the training data too well, capturing
not just the underlying patterns but also the noise and random fluctuations
specific to th...
  2. The tension between overfitting and underfitting reflects the fundamental
bias-variance tradeoff in machine learning. Simpler models exhibit high bias
(systematic errors from overly restrictive assump...
  3. The train/test split is a fundamental practice for honest evaluation. Before any
model training begins, a portion of the data (typically 20-30%) is set aside as
a test set that the model will never se...
  4. Regularization techniques constrain model complexity to reduce overfitting
without sacrificing the model's ability to learn meaningful patterns. L1
regularization (also known as Lasso) adds a penalty ...
  5. ## Practical Considerations

Training neural networks presents several practical challenges beyond
architecture design. Overfitting is a persistent concern, es


## Zusammenfassung

- **RAG mit LangChain**: Wenige Zeilen Code für ein vollständiges System
- **Pipeline**: Dokumente -> Chunks -> Qdrant (Hybride Suche) -> Retriever -> LLM
- **LCEL**: Verkettung mit `|` Operator, parallele Ketten mit Dictionaries
- **Prompt Engineering**: "Sage, wenn du es nicht weißt" verhindert Halluzinationen
- **Retrieval-Parameter**: `k` und `search_type` steuern Kontext-Qualität

**Jetzt sind Sie bereit, Ihr eigenes RAG-System zu bauen!**